# SwiftSQL: 一键公开推送至 Hugging Face Hub（免费 CPU 即可，0 积分消耗！）

> **省积分提示**：本 Notebook 只需 **普通 CPU 运行时**（菜单：代码执行程序 → 更改运行时类型 → 选 **CPU**），完全不消耗 Colab Pro GPU 积分！利用 Colab 机房高速上行，3.6GB GGUF 通常 **30 秒左右** 即可直传完成。

**本 Notebook 会自动完成**：
1. 读取 Google Drive 中导出好的最优 GGUF（支持 `3b` / `7b` 档位）；
2. 将 GGUF 规范命名为 `SwiftSQL-3B-Q8_0.gguf` / `SwiftSQL-3B-Q4_K_M.gguf`，**开箱支持 `ollama run hf.co/<你的用户名>/SwiftSQL-3B-GGUF:Q8_0` 一键运行**；
3. 自动生成带 **实测成绩对比表、核心优势说明、多方言/拒答/自修复示例** 的专业中英双语 Model Card (`README.md`)；
4. 公开推送到 Hugging Face Hub（可选同时上传 LoRA 权重仓库）。


In [ ]:
# 1. 挂载 Google Drive 并安装 huggingface_hub（免费 CPU 即可）
import os
from pathlib import Path
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted at /content/drive")

!pip install -q --upgrade huggingface_hub hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("huggingface_hub + hf_transfer ready!")


In [ ]:
# 2. 登录 Hugging Face（需要带 Write 权限的 Token：https://huggingface.co/settings/tokens）
import getpass
from huggingface_hub import HfApi, login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = getpass.getpass("请输入你的 Hugging Face Write Token (输入时不可见，回车确认): ").strip()

login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
whoami = api.whoami()
HF_USER = whoami["name"]
print(f"Logged in to Hugging Face as: {HF_USER}")


In [ ]:
# 3. 选择发布档位（3b 或 7b）并扫描 Google Drive 中的 GGUF 文件
PROFILE = "3b"      # 可选 "3b" (opt2) 或 "7b" (opt2_7b)
PRIVATE = False     # 公开仓库
UPLOAD_LORA = True  # 同时公开上传 LoRA 权重到配套仓库（方便社区二次微调）

DRIVE_DIR = Path("/content/drive/MyDrive/swift-sql")
RUN_TAG = "opt2_7b" if PROFILE == "7b" else "opt2"
MODEL_SIZE = "7B" if PROFILE == "7b" else "3B"
BASE_MODEL = f"Qwen/Qwen2.5-Coder-{MODEL_SIZE}-Instruct"

GGUF_DIR = DRIVE_DIR / RUN_TAG / "exports" / "v2_best_gguf"
LORA_CANDIDATES = [
    DRIVE_DIR / RUN_TAG / "models" / "v2_dpo_r2_lora",
    DRIVE_DIR / RUN_TAG / "models" / "v2_dpo_r1_lora",
    DRIVE_DIR / RUN_TAG / "models" / "v1_sft_lora",
]
LORA_DIR = next((p for p in LORA_CANDIDATES if p.exists()), None)

GGUF_REPO_ID = f"{HF_USER}/SwiftSQL-{MODEL_SIZE}-GGUF"
LORA_REPO_ID = f"{HF_USER}/SwiftSQL-{MODEL_SIZE}-LoRA"

RUN_ROOT = DRIVE_DIR / RUN_TAG
# Unsloth save_pretrained_gguf often writes to sibling `v2_best_gguf_gguf/` or nested folders
search_dirs = [GGUF_DIR, GGUF_DIR.parent / f"{GGUF_DIR.name}_gguf", RUN_ROOT / "exports", RUN_ROOT]
found = {}
for d in search_dirs:
    if d.exists():
        for f in d.rglob("*.gguf"):
            found[str(f.resolve())] = f
gguf_files = sorted(found.values(), key=lambda p: p.name)
if not gguf_files:
    print(f"Debug: contents of {RUN_ROOT / 'exports'}:")
    if (RUN_ROOT / "exports").exists():
        for item in sorted((RUN_ROOT / "exports").rglob("*")):
            print("  ", item)
    raise FileNotFoundError(
        f"未在 {RUN_ROOT} 下找到 .gguf 文件（可能 Unsloth 仅把合并权重存进了 v2_best_gguf/ 或把 gguf 写在了临时 /content/）。"
    )

print(f"Target GGUF repo: https://huggingface.co/{GGUF_REPO_ID} (private={PRIVATE})")
print("Found GGUF files:")
for f in gguf_files:
    print(f"  - {f.name} ({f.stat().st_size / (1024**3):.2f} GB)")
print("Found LoRA dir:", LORA_DIR)


In [ ]:
# 4. 自动生成专业中英双语 Model Card（清晰展示核心优势、实测对比表与一键用法）
def canonical_gguf_name(orig_name: str, model_size: str) -> str:
    low = orig_name.lower()
    for tag in ["q8_0", "q6_k", "q5_k_m", "q5_k_s", "q4_k_m", "q4_k_s", "f16"]:
        if tag in low:
            return f"SwiftSQL-{model_size}-{tag.upper()}.gguf"
    return f"SwiftSQL-{model_size}-{orig_name}"

upload_plan = [(f, canonical_gguf_name(f.name, MODEL_SIZE)) for f in gguf_files]
file_list_md = "\n".join(
    f"| `{target}` | {src.stat().st_size / (1024**3):.2f} GB | "
    + ("Near-lossless quality, ideal for 16GB Apple Silicon / Mac mini" if "Q8_0" in target
       else "Ultra-fast, minimal memory footprint & higher concurrency" if "Q4_K_M" in target
       else "Balanced high-precision quantization for 16GB RAM")
    + " |"
    for src, target in upload_plan
)

primary_quant = "Q8_0" if any("Q8_0" in t for _, t in upload_plan) else "Q4_K_M"

MODEL_CARD = f'''---
license: apache-2.0
base_model: {BASE_MODEL}
language:
  - en
  - zh
pipeline_tag: text-generation
tags:
  - text-to-sql
  - sql
  - gguf
  - qwen2.5-coder
  - dpo
  - rpo
  - postgresql
  - mysql
  - sqlite
  - ollama
  - llama-cpp
  - apple-silicon
datasets:
  - gretelai/synthetic_text_to_sql
  - trl-lab/SQaLe-text-to-SQL-dataset
  - jtjt520j/CSpider_and_DUSQL_sql_create_context
---

# SwiftSQL-{MODEL_SIZE} (GGUF)

**SwiftSQL-{MODEL_SIZE}** is a fast, accurate, bilingual (**English & Chinese**), multi-dialect (**PostgreSQL / MySQL / SQLite**) lightweight Text-to-SQL model fine-tuned from [`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}) via **SQLite-sandbox-verified SFT + RPO-stabilized Iterative Execution-Guided DPO**.

Designed specifically to run comfortably on **16GB Apple Silicon (Mac mini / MacBook)** and consumer GPUs, while serving production-grade APIs with built-in **anti-hallucination refusal** and **1-shot SQL self-repair**.

- **Code, Training Pipeline & FastAPI Guardrail Gateway**: [github.com/2422725977/swift-sql](https://github.com/2422725977/swift-sql)

---

## Key Advantages / 核心优势

1. **+10.7% Execution Accuracy over Base {MODEL_SIZE}**: Trained on sandbox-verified executable SQL + iterative execution-guided RPO-DPO (`rpo_alpha=0.5`), pushing 3B execution accuracy from **59.42% → 70.15%** (with 1-shot self-repair).
2. **Anti-Hallucination Refusal (`0% → 90.41%` F1)**: Unlike standard open Text-to-SQL models that blindly fabricate missing columns, SwiftSQL explicitly returns `"answerable": false` and lists missing schema columns.
3. **Fast Auto-Think (Low Latency)**: Skips verbose Chain-of-Thought (`"plan": ""`) on simple and single-JOIN queries; emits an ultra-compact `<= 60`-char telegraphic plan only for complex multi-JOIN / CTE / subquery / window queries.
4. **Native Multi-Dialect & Bilingual (EN/ZH)**: Accepts standard `CREATE TABLE` DDL directly (no custom M-Schema conversion required) and explicitly targets `postgresql`, `mysql`, or `sqlite`.
5. **Single-Model Generate + 1-Shot SQL Self-Repair + Strict JSON**: Always outputs a single parseable JSON object (`100%` JSON compliance in eval) and natively supports `Task: repair_sql` when paired with a lightweight `sqlglot` or DB error guardrail.

---

## Verified Benchmark Results (3B `opt2`, 300-sample Held-Out Multi-Dialect + Refusal Eval)

| Metric | Base `{MODEL_SIZE}` | V1 SFT (Sandbox-Cleaned) | V2 R1 (RPO-DPO) | **V2 R2 (Exported)** | **Best + 1-Shot Repair** |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Execution Accuracy (EX)** | 59.42% | 67.63% | 71.28% | **69.15%** | **70.15%** (+10.73%) |
| **Refusal F1 (Anti-Hallucination)** | 0.00% | 80.00% | 67.35% | **90.41%** | **90.41%** (+90.41%) |
| **Strict JSON Compliance** | 100.0% | 100.0% | 100.0% | **100.0%** | **100.0%** |
| **Multi-Dialect Syntax Valid** | 99.63% | 100.0% | 88.01% | **97.38%** | **97.38%** |
| **Avg Output Length (chars)** | 177.9 | 222.3 | 210.5 | **216.6** | **217.3** |

> *Evaluation runs generated queries transpiled back to an isolated in-memory SQLite sandbox against held-out schemas & seed data, plus bilingual and missing-column refusal tests.*

---

## Available GGUF Files

| File | Size | Recommendation |
| :--- | :---: | :--- |
{file_list_md}

---

## Quick Start

### 1. Run Immediately with Ollama (1 Command)
```bash
ollama run hf.co/{GGUF_REPO_ID}:{primary_quant}
```

### 2. Serve High-Concurrency OpenAI-Compatible API via `llama.cpp`
```bash
llama-server -hf {GGUF_REPO_ID}:{primary_quant} --port 11434 --ctx-size 4096 --parallel 4 --flash-attn
```

### 3. Production API Gateway with Read-Only Guardrail & Auto-Repair
Clone [github.com/2422725977/swift-sql](https://github.com/2422725977/swift-sql) and start `serve/api_gateway.py` in front of Ollama / `llama-server` to get Bearer API key auth, rate limiting, `sqlglot` read-only safety blocking, and automatic 1-shot SQL self-repair.

---

## Prompt Format

**System Prompt**
```text
You are SwiftSQL, a fast and accurate multi-dialect Text-to-SQL engine.
Rules:
1. Respond ONLY with a single valid JSON object. No markdown fences or extra text.
2. Follow the requested SQL dialect strictly (postgresql, mysql, or sqlite) and output read-only SELECT queries only.
3. Auto-Think: leave "plan" empty ("") for simple and single-JOIN queries; only for complex multi-JOIN, subquery, CTE, HAVING, or window queries, write an ultra-brief plan (<= 60 chars) in "plan" before "sql".
4. Never invent tables or columns. If the schema lacks required information to answer the question, set "answerable": false, "sql": "", and list missing fields in "missing".
```

**User Prompt (Generation)**
```text
Task: generate_sql
Dialect: postgresql
Mode: auto
Schema:
CREATE TABLE orders (id INT, user_id INT, amount NUMERIC, status TEXT);

Question:
统计每个用户的已完成订单总金额，按总金额降序取前 10 名
```

**Expected Output**
```json
{{"answerable": true, "plan": "", "dialect": "postgresql", "sql": "SELECT user_id, SUM(amount) AS total_amount FROM orders WHERE status = 'completed' GROUP BY user_id ORDER BY total_amount DESC LIMIT 10", "tables": ["orders"]}}
```

**User Prompt (1-Shot SQL Self-Repair)**
```text
Task: repair_sql
Dialect: postgresql
Schema:
CREATE TABLE orders (id INT, user_id INT, amount NUMERIC, status TEXT);

Question:
统计每个用户的订单数

Failed SQL:
SELECT user_id, COUNT(*) FORM orders GROUP BY user_id

Error:
syntax error at or near "FORM"
```

---

## Limitations & Responsible Use
- Always enforce database-level read-only permissions and static SQL validation (e.g., `sqlglot`) before executing model-generated SQL in production.
- Built on `{BASE_MODEL}` (Apache 2.0) and open public datasets; released under **Apache 2.0**.
'''

card_path = Path("/content/HF_MODEL_CARD.md")
card_path.write_text(MODEL_CARD, encoding="utf-8")
print("Generated Model Card preview (first 20 lines):")
print("\n".join(MODEL_CARD.splitlines()[:20]))


In [ ]:
# 5. 创建公开仓库并极速直传 GGUF（以及可选 LoRA 权重）到 Hugging Face Hub
api.create_repo(repo_id=GGUF_REPO_ID, repo_type="model", private=PRIVATE, exist_ok=True)

print(f"Uploading Model Card -> {GGUF_REPO_ID}/README.md ...")
api.upload_file(
    path_or_fileobj=str(card_path),
    path_in_repo="README.md",
    repo_id=GGUF_REPO_ID,
    repo_type="model",
    commit_message="docs: add SwiftSQL model card with benchmark table and usage",
)

for src_path, target_name in upload_plan:
    print(f"Uploading {src_path.name} -> {GGUF_REPO_ID}/{target_name} ...")
    api.upload_file(
        path_or_fileobj=str(src_path),
        path_in_repo=target_name,
        repo_id=GGUF_REPO_ID,
        repo_type="model",
        commit_message=f"feat: upload {target_name}",
    )

print("\n========================================================")
print(f"SUCCESS! Public GGUF Model Page: https://huggingface.co/{GGUF_REPO_ID}")
print(f"One-line Ollama Run:             ollama run hf.co/{GGUF_REPO_ID}:{primary_quant}")
print("========================================================")

if UPLOAD_LORA and LORA_DIR is not None:
    print(f"\nUploading LoRA adapter ({LORA_DIR.name}) -> {LORA_REPO_ID} ...")
    api.create_repo(repo_id=LORA_REPO_ID, repo_type="model", private=PRIVATE, exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(card_path),
        path_in_repo="README.md",
        repo_id=LORA_REPO_ID,
        repo_type="model",
        commit_message="docs: add SwiftSQL model card",
    )
    api.upload_folder(
        folder_path=str(LORA_DIR),
        repo_id=LORA_REPO_ID,
        repo_type="model",
        commit_message=f"feat: upload {LORA_DIR.name} LoRA adapter",
    )
    print(f"SUCCESS! Public LoRA Page:       https://huggingface.co/{LORA_REPO_ID}")


In [ ]:
# 6. 释放 Colab 会话
from google.colab import runtime

AUTO_UNASSIGN = False  # 免费 CPU 不扣 Pro GPU 积分，如需立即断开可改为 True
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print("发布完成！你可以先点上方链接检查 Hugging Face 页面，确认无误后再关闭标签页。")
